In [8]:
import glob
from langchain_community.document_loaders import DirectoryLoader, TextLoader
import pathlib
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import tiktoken

In [ ]:
files = glob.glob("rag_mock_data/**/*.*", recursive=True)

entire_knowlege_basee = ""
print(f"Found {len(files)} files in the knowledge base")
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        entire_knowlege_basee += f.read()
        entire_knowlege_basee += "\n\n"
print(f"Length of the entire knowledge base is {len(entire_knowlege_basee)}")

Found 18 files in the knowledge base
Length of the entire knowledge base is 39022


In [6]:
entire_knowlege_basee

'# RAG Mock Dataset\n\nThis is a synthetic dataset generated for testing a Retrieval Augmented Generation (RAG) pipeline. All content is fictional. It\'s designed to mimic a real company\'s internal knowledge base, with variety across topics, file formats, document length, and directory depth, so you can test chunking, embedding, retrieval, and citation behavior.\n\n## Structure\n\n```\nrag_mock_data/\n├── hr/\n│   ├── policies/\n│   │   ├── remote_work_policy.md\n│   │   └── paid_time_off.md\n│   ├── benefits/\n│   │   ├── health_insurance_guide.md\n│   │   └── retirement_401k.txt\n│   └── employee_directory_sample.json\n├── engineering/\n│   ├── architecture/\n│   │   ├── system_architecture_overview.md\n│   │   └── adr_003_message_queue_choice.md\n│   └── runbooks/\n│       ├── incident_response_runbook.md\n│       ├── database_migration_runbook.md\n│       └── oncall/\n│           └── escalation_matrix.md\n├── product/\n│   └── specs/\n│       ├── saved_carts_feature_spec.md\n│    

In [9]:
encoding = tiktoken.encoding_for_model("gpt-4.1-nano")
token = encoding.encode(entire_knowlege_basee)
print(f"The total length of token {len(token)}")

The total length of token 8345


In [ ]:
documents = []
folders = glob.glob("rag_mock_data/*")
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder, glob="**/*.*", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
    )
    docc = loader.load()
    for doc in docc:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)
print(f"The total documents in this are {len(documents)}")

The total documents in this are 17


In [17]:
chunk = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
chunking = chunk.split_documents(documents)
print(f"The total chunks: {len(chunking)}")
chunking[0]

The total chunks: 98


Document(metadata={'source': 'rag_mock_data\\engineering\\architecture\\adr_003_message_queue_choice.md', 'doc_type': 'engineering'}, page_content='# ADR 003: Choice of Message Queue Technology\n\n**Status:** Accepted\n**Date:** 2024-08-12\n**Deciders:** Platform Architecture Team\n\n## Context\n\nAs the platform grew past 12 microservices, synchronous REST calls between services created tight coupling and cascading failure risk during traffic spikes. We needed an asynchronous messaging backbone to decouple services, support event driven workflows, and provide durable delivery guarantees.\n\n## Decision')